# BDD100K referee benchmark: YOLO vs Grounding DINO vs Florence-2

Notebook này chạy ba model trên cùng 100 ảnh, tính COCO mAP/recall và đo latency, FPS, peak VRAM. Model được tải từ Hugging Face **chỉ khi cell inference chạy**.

Trước khi chạy: chọn **Runtime > Change runtime type > T4 GPU**. Sau đó chạy từng cell từ trên xuống. Bạn chỉ cần upload file `bdd100k_ground_truth.json` khi notebook yêu cầu.

In [ ]:
# Cell 1 - cài dependencies vào runtime Colab tạm thời
%pip install -q "transformers==4.56.2" "ultralytics>=8.3,<9" "pycocotools>=2.0.7,<3" "huggingface-hub>=0.35,<2" einops timm
import os, torch
assert torch.cuda.is_available(), 'Hãy bật T4 GPU: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

In [ ]:
# Cell 2 - tải 100 ảnh từ repository và upload ground truth
from google.colab import files
from pathlib import Path
import json, shutil, zipfile, requests

ROOT = Path('/content/model_qc_benchmark')
IMAGES = ROOT / 'images'
RESULTS = ROOT / 'results'
IMAGES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
zip_path = ROOT / 'images.zip'
url = 'https://github.com/NhatTruong1905/cvat-project-v2/raw/refs/heads/develop/data/bdd100k_mvp_100_seed42/bdd100k_mvp_100_seed42_images.zip'
if not zip_path.exists():
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
if not any(IMAGES.iterdir()):
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(IMAGES)
print('Đã chuẩn bị', len(list(IMAGES.glob('*.jpg'))), 'ảnh')
print('Chọn file: data/bdd100k_mvp_100_seed42/bdd100k_ground_truth.json')
uploaded = files.upload()
gt_name = next((name for name in uploaded if name.endswith('.json')), None)
assert gt_name, 'Chưa upload file JSON ground truth'
GT_PATH = ROOT / 'bdd100k_ground_truth.json'
GT_PATH.write_bytes(uploaded[gt_name])
ground_truth = json.loads(GT_PATH.read_text())
assert len(ground_truth) == 100
print('Ground truth:', len(ground_truth), 'frames,', sum(len(x['boxes']) for x in ground_truth), 'boxes')

In [ ]:
# Cell 3 - hàm inference chung và ba model adapters
import gc, platform, statistics, time
from PIL import Image

DEVICE = 'cuda:0'
CLASSES = ['pedestrian','rider','car','truck','bus','train','motorcycle','bicycle','traffic light','traffic sign']
ALIASES = {'person':'pedestrian','people':'pedestrian','bike':'bicycle','motorbike':'motorcycle','motor':'motorcycle','stop sign':'traffic sign','street sign':'traffic sign'}
IMAGE_PATHS = sorted(IMAGES.glob('*.jpg'))

def label_bdd(value):
    value = ALIASES.get(str(value).lower().strip().strip('. '), str(value).lower().strip().strip('. '))
    return value if value in CLASSES else None

def save_run(name, role, model_id, predict, warmup=2):
    for path in IMAGE_PATHS[:warmup]: predict(path)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    predictions, latencies = [], []
    for frame, path in enumerate(IMAGE_PATHS):
        torch.cuda.synchronize(); started = time.perf_counter()
        detections = predict(path)
        torch.cuda.synchronize(); latencies.append(time.perf_counter() - started)
        for index, (label, score, box) in enumerate(detections):
            predictions.append({'frame':frame,'id':f'{name}-{frame}-{index}','image':path.name,'label':label,'confidence':round(float(score),6),'bbox':[round(float(x),3) for x in box]})
        if (frame + 1) % 10 == 0: print(name, frame + 1, '/', len(IMAGE_PATHS))
    runtime = {'model':name,'role':role,'model_id':model_id,'device':torch.cuda.get_device_name(0),'images':len(IMAGE_PATHS),'detections':len(predictions),'total_seconds':round(sum(latencies),4),'mean_ms':round(statistics.mean(latencies)*1000,2),'median_ms':round(statistics.median(latencies)*1000,2),'p95_ms':round(sorted(latencies)[int(len(latencies)*.95)-1]*1000,2),'fps':round(len(IMAGE_PATHS)/sum(latencies),3),'peak_gpu_memory_mb':round(torch.cuda.max_memory_allocated()/2**20,1)}
    (RESULTS/f'{name}_predictions.json').write_text(json.dumps({'predictions':predictions}, indent=2))
    (RESULTS/f'{name}_runtime.json').write_text(json.dumps(runtime, indent=2))
    print(json.dumps(runtime, indent=2)); return runtime

def release(*objects):
    for obj in objects: del obj
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 4 - baseline YOLO-BDD100K
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
YOLO_ID = 'dronefreak/bdd100k-yolov8s'
weights = hf_hub_download(YOLO_ID, 'best.pt')
yolo = YOLO(weights)
def predict_yolo(path):
    result = yolo.predict(str(path), conf=.01, imgsz=960, device=DEVICE, max_det=300, verbose=False)[0]
    found = []
    for box in result.boxes:
        label = label_bdd(result.names[int(box.cls.item())])
        if label: found.append((label, box.conf.item(), box.xyxy[0].tolist()))
    return found
save_run('yolo_bdd100k', 'baseline', YOLO_ID, predict_yolo)
del yolo; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 5 - candidate Grounding DINO
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
DINO_ID = 'IDEA-Research/grounding-dino-tiny'
dino_processor = AutoProcessor.from_pretrained(DINO_ID)
# Grounding DINO tiny chạy FP32: một số tensor nội bộ luôn là FP32 và sẽ lỗi nếu weights là FP16.
dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID, torch_dtype=torch.float32).to(DEVICE).eval()
dino_prompt = '. '.join(CLASSES) + '.'
def predict_dino(path):
    image = Image.open(path).convert('RGB')
    inputs = dino_processor(images=image, text=dino_prompt, return_tensors='pt').to(DEVICE)
    inputs['pixel_values'] = inputs['pixel_values'].to(dtype=dino.dtype)
    with torch.inference_mode(): outputs = dino(**inputs)
    try:
        result = dino_processor.post_process_grounded_object_detection(outputs, inputs.input_ids, box_threshold=.01, text_threshold=.20, target_sizes=[image.size[::-1]])[0]
    except TypeError:
        result = dino_processor.post_process_grounded_object_detection(outputs, inputs.input_ids, threshold=.01, text_threshold=.20, target_sizes=[image.size[::-1]])[0]
    labels = result.get('text_labels', result.get('labels', [])); found = []
    for label, score, box in zip(labels, result['scores'], result['boxes']):
        label = label_bdd(label)
        if label: found.append((label, score.item(), box.tolist()))
    return found
save_run('grounding_dino', 'candidate', DINO_ID, predict_dino)
del dino, dino_processor; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 6 - candidate Florence-2
from transformers import AutoProcessor, Florence2ForConditionalGeneration
# Checkpoint đã được chuyển đổi cho Florence-2 native trong Transformers 4.56.2.
FLORENCE_ID = 'florence-community/Florence-2-base-ft'
florence_processor = AutoProcessor.from_pretrained(FLORENCE_ID)
florence = Florence2ForConditionalGeneration.from_pretrained(FLORENCE_ID, torch_dtype=torch.float16, attn_implementation='eager').to(DEVICE).eval()
def predict_florence(path):
    image = Image.open(path).convert('RGB'); task = '<OD>'
    inputs = florence_processor(text=task, images=image, return_tensors='pt').to(DEVICE, torch.float16)
    with torch.inference_mode():
        ids = florence.generate(input_ids=inputs['input_ids'], pixel_values=inputs['pixel_values'], max_new_tokens=1024, num_beams=3, do_sample=False)
    text = florence_processor.batch_decode(ids, skip_special_tokens=False)[0]
    parsed = florence_processor.post_process_generation(text, task=task, image_size=image.size)[task]
    found = []
    for label, box in zip(parsed.get('labels', []), parsed.get('bboxes', [])):
        label = label_bdd(label)
        if label: found.append((label, 1.0, box))
    return found
save_run('florence2', 'candidate', FLORENCE_ID, predict_florence)
del florence, florence_processor; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 7 - COCO evaluation, leaderboard và tải results.zip
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
categories = [{'id':i+1,'name':name} for i,name in enumerate(CLASSES)]
cat_ids = {x['name']:x['id'] for x in categories}
images, annotations, ann_id = [], [], 1
for image_id, frame in enumerate(ground_truth, 1):
    images.append({'id':image_id,'file_name':frame['filename'],'width':frame['width'],'height':frame['height']})
    for item in frame['boxes']:
        x1,y1,x2,y2 = item['bbox']; w,h=max(0,x2-x1),max(0,y2-y1)
        annotations.append({'id':ann_id,'image_id':image_id,'category_id':cat_ids[item['class']],'bbox':[x1,y1,w,h],'area':w*h,'iscrowd':0}); ann_id += 1
gt_data = {'info':{},'licenses':[],'images':images,'annotations':annotations,'categories':categories}
image_ids = {x['file_name']:x['id'] for x in images}
coco_gt = COCO(); coco_gt.dataset=gt_data; coco_gt.createIndex()
metric_names=['mAP_50_95','AP50','AP75','AP_small','AP_medium','AP_large','AR_1','AR_10','AR_100','AR_small','AR_medium','AR_large']
rows=[]
for name in ['yolo_bdd100k','grounding_dino','florence2']:
    raw=json.loads((RESULTS/f'{name}_predictions.json').read_text())['predictions']; detections=[]
    for item in raw:
        if item['image'] not in image_ids or item['label'] not in cat_ids: continue
        x1,y1,x2,y2=item['bbox']; detections.append({'image_id':image_ids[item['image']],'category_id':cat_ids[item['label']],'bbox':[x1,y1,max(0,x2-x1),max(0,y2-y1)],'score':item['confidence']})
    coco_dt=coco_gt.loadRes(detections); evaluator=COCOeval(coco_gt,coco_dt,'bbox'); evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
    metrics={key:round(float(value),6) for key,value in zip(metric_names,evaluator.stats)}; metrics.update({'ground_truth_boxes':len(annotations),'predicted_boxes':len(detections)})
    (RESULTS/f'{name}_metrics.json').write_text(json.dumps(metrics,indent=2))
    runtime=json.loads((RESULTS/f'{name}_runtime.json').read_text()); rows.append({**runtime,**metrics})
rows.sort(key=lambda x:x['mAP_50_95'], reverse=True)
(RESULTS/'leaderboard.json').write_text(json.dumps(rows,indent=2))
header='| Model | Role | mAP50-95 | AP50 | AP75 | AR@100 | Mean ms | FPS | Peak VRAM MB |\n|---|---|---:|---:|---:|---:|---:|---:|---:|'
lines=[header]+[f"| {r['model']} | {r['role']} | {r['mAP_50_95']:.4f} | {r['AP50']:.4f} | {r['AP75']:.4f} | {r['AR_100']:.4f} | {r['mean_ms']:.2f} | {r['fps']:.2f} | {r['peak_gpu_memory_mb']} |" for r in rows]
leaderboard='\n'.join(lines)+'\n'; (RESULTS/'leaderboard.md').write_text(leaderboard); print(leaderboard)
archive=shutil.make_archive('/content/model_qc_benchmark_results','zip',RESULTS)
files.download(archive)

## Lưu ý khi đọc kết quả

- So sánh chất lượng chính bằng `mAP_50_95`, `AP50`, `AP75` và `AR_100`.
- So sánh vận hành bằng `mean_ms`, `fps` và `peak_gpu_memory_mb`.
- Florence-2 `<OD>` không trả confidence từng box nên notebook gán `1.0`. Vì vậy AP theo ranking và khả năng dùng trực tiếp cho Probability Differential cần được xem là hạn chế của model, không phải confidence đã được hiệu chỉnh.
- Không thay threshold dựa trên chính 100 ảnh test này. Nếu tuning, cần dùng validation split riêng.